# 01 — Data quality and provenance

This notebook loads the local IHK Berlin export, checks its schema and quality, and records caveats before analysis. It does not silently remove duplicates.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from berlin_businesses.config import DATA_RAW, DATA_PROCESSED
from berlin_businesses.data import load_sources, validate_schema, coerce_coordinates

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
print('Project root:', ROOT)
print('Raw data folder:', DATA_RAW)

In [ ]:
df = coerce_coordinates(load_sources(DATA_RAW))
df.shape

In [ ]:
display(df.head())
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'missing_pct': (100*df.isna().mean()).round(3)}).sort_values('missing_pct', ascending=False))

In [ ]:
validation = validate_schema(df)
print('Rows:', validation['rows'])
print('Columns:', validation['columns'])
print('Exact duplicate rows:', validation['duplicate_rows'])
print('Missing coordinate columns:', validation['missing_coordinate_columns'])

## Coordinate plausibility

The following bounds are only a rough plausibility check for Berlin, not an authoritative city boundary. Do not drop records solely on this check without reviewing them.

In [ ]:
if {'latitude', 'longitude'}.issubset(df.columns):
    bounds = df['latitude'].between(52.3, 52.7) & df['longitude'].between(13.0, 13.8)
    display(pd.Series({'records': len(df), 'inside_rough_bounds': int(bounds.sum()), 'outside_or_missing': int((~bounds).sum())}))
else:
    print('Coordinate columns are missing; check the source schema.')

## Save a cleaned working snapshot

This step converts coordinates to numeric values but does not deduplicate rows or overwrite raw files. Review whether the export contains monthly snapshots before combining multiple months; repeated businesses across months are not necessarily duplicate businesses.

In [ ]:
df.to_parquet(DATA_PROCESSED / 'businesses_loaded.parquet', index=False)
print('Saved:', DATA_PROCESSED / 'businesses_loaded.parquet')

## Interpretation checklist

- Confirm source URL, download date, and terms in `data/README.md`.
- Determine whether each file is a monthly snapshot or a distinct incremental extract.
- Define a stable unit of analysis before deduplicating.
- Treat missingness and geocoding precision as potential sources of bias.